# 04 — GroupBy, Aggregation, and Window Operations

Split-apply-combine, the `agg`/`transform`/`filter` distinction, `pivot_table`, rolling/expanding windows, and the classic top-N-per-group problem — pandas' answer to Spark's window functions.

In [1]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "region": ["east", "east", "east", "east", "west", "west", "west"],
    "rep": ["alice", "alice", "bob", "bob", "cara", "cara", "dan"],
    "month": ["2024-01", "2024-02", "2024-01", "2024-02", "2024-01", "2024-02", "2024-01"],
    "amount": [100, 150, 200, 90, 300, 250, 120],
})

## 1. Split-apply-combine

`df.groupby(key)` splits the DataFrame into groups by key, and every subsequent operation (`.sum()`, `.agg()`, `.apply()`) applies per-group, then combines results back into one object. This is conceptually identical to Spark's `groupBy`/window-partition model — the difference is pandas does it in-memory, single-machine.

In [2]:
region_summary = sales.groupby("region").agg(
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
    max_amount=("amount", "max"),
    n_rows=("amount", "count"),
).reset_index()
region_summary

,region,total_amount,avg_amount,max_amount,n_rows
0,east,540,135.000000,200,4
1,west,670,223.333333,300,3


## 2. `agg` vs `transform` vs `filter` — the distinction interviewers probe

- **`.agg()`** — one row **per group** out (a reduction, like SQL `GROUP BY`).
- **`.transform()`** — **same number of rows as the input**, each row gets its group's computed value broadcast back — the pandas equivalent of a window function with no explicit frame (like `SUM(amount) OVER (PARTITION BY region)` in SQL/Spark).
- **`.filter()`** — keeps or drops **entire groups** based on a group-level condition (e.g. "only reps with more than 1 sale").

In [3]:
# transform: broadcast each region's total back onto every row of that region
sales["region_total"] = sales.groupby("region")["amount"].transform("sum")
sales["pct_of_region"] = (sales["amount"] / sales["region_total"] * 100).round(1)
sales

,region,rep,month,amount,region_total,pct_of_region
0,east,alice,2024-01,100,540,18.5
1,east,alice,2024-02,150,540,27.8
2,east,bob,2024-01,200,540,37.0
3,east,bob,2024-02,90,540,16.7
4,west,cara,2024-01,300,670,44.8
5,west,cara,2024-02,250,670,37.3
6,west,dan,2024-01,120,670,17.9


In [4]:
# filter: keep only reps with more than one sale row
sales.groupby("rep").filter(lambda g: len(g) > 1)

,region,rep,month,amount,region_total,pct_of_region
0,east,alice,2024-01,100,540,18.5
1,east,alice,2024-02,150,540,27.8
2,east,bob,2024-01,200,540,37.0
3,east,bob,2024-02,90,540,16.7
4,west,cara,2024-01,300,670,44.8
5,west,cara,2024-02,250,670,37.3


## 3. Ranking within groups — pandas' window-function equivalent

`groupby(...)["col"].rank(method=..., ascending=...)` mirrors SQL/Spark's `RANK()`/`DENSE_RANK()`/`ROW_NUMBER()` exactly via the `method` argument:

- `method="first"` → `ROW_NUMBER()` (unique, ties broken by original order)
- `method="min"` → `RANK()` (ties share the lower rank, next rank skips)
- `method="dense"` → `DENSE_RANK()` (ties share the rank, no skip)

In [5]:
rep_totals = sales.groupby(["region", "rep"], as_index=False)["amount"].sum().rename(columns={"amount": "total"})

rep_totals["row_number"] = rep_totals.groupby("region")["total"].rank(method="first", ascending=False)
rep_totals["rank"] = rep_totals.groupby("region")["total"].rank(method="min", ascending=False)
rep_totals["dense_rank"] = rep_totals.groupby("region")["total"].rank(method="dense", ascending=False)
rep_totals.sort_values(["region", "row_number"])

,region,rep,total,row_number,rank,dense_rank
1,east,bob,290,1.0,1.0,1.0
0,east,alice,250,2.0,2.0,2.0
2,west,cara,550,1.0,1.0,1.0
3,west,dan,120,2.0,2.0,2.0


## 4. Classic problem: top-N per group

"For each region, find the rep with the highest total sales." Same problem, same shape of solution as the Spark notebooks — rank within each group, then filter.

In [6]:
top_rep_per_region = (
    rep_totals[rep_totals["row_number"] == 1]
    .drop(columns=["row_number", "rank", "dense_rank"])
    .reset_index(drop=True)
)
top_rep_per_region
# generalizes to top-3 per group: rep_totals[rep_totals["row_number"] <= 3]

,region,rep,total
0,east,bob,290
1,west,cara,550


## 5. `pivot_table` and `crosstab`

`pivot_table` reshapes long data into wide (like a spreadsheet pivot — same operation as Spark's `groupBy().pivot()` from the Spark notebooks). `crosstab` is a shorthand specifically for frequency counts across two categorical columns.

In [7]:
pivoted = sales.pivot_table(index="rep", columns="month", values="amount", aggfunc="sum", fill_value=0)
pivoted

month,2024-01,2024-02
rep,,
alice,100,150
bob,200,90
cara,300,250
dan,120,0


## 6. Rolling and expanding windows

- **`.rolling(window=n)`** — a fixed-size sliding window (e.g. 3-period moving average). Combine with `.groupby()` for a per-group rolling calculation.
- **`.expanding()`** — a growing window from the start of the group/series to the current row — the standard way to compute a running total or cumulative average.
- **`.cumsum()`/`.cumcount()`/`.cummax()`** — dedicated cumulative methods, generally clearer and faster than `.expanding().sum()` when you just need a running total.

In [8]:
ts = pd.DataFrame({
    "rep": ["alice"] * 5,
    "day": pd.date_range("2024-01-01", periods=5),
    "amount": [10, 20, 15, 30, 25],
}).set_index("day")

ts["rolling_3d_avg"] = ts["amount"].rolling(window=3).mean()
ts["running_total"] = ts.groupby("rep")["amount"].cumsum()
ts

,rep,amount,rolling_3d_avg,running_total
day,,,,
2024-01-01,alice,10,NaN,10
2024-01-02,alice,20,NaN,30
2024-01-03,alice,15,15.000000,45
2024-01-04,alice,30,21.666667,75
2024-01-05,alice,25,23.333333,100


## 7. Interview Q&A

1. **"How do you add each group's total as a column on every row, without collapsing the DataFrame?"** — `df.groupby(key)["col"].transform("sum")`; `.agg()` would collapse to one row per group instead.
2. **"How do you get the top-N rows per group?"** — `groupby(key)["metric"].rank(method="first", ascending=False)` (or `.rank("min")` if ties should all be included), then filter `<= N`.
3. **"Difference between `.rank(method="min")` and `method="dense"`?"** — `min` leaves a gap after ties (like SQL `RANK()`), `dense` doesn't (like `DENSE_RANK()`) — identical semantics to the Spark/SQL versions of the same functions.
4. **"When would you use `.rolling()` vs `.expanding()`?"** — `.rolling(n)` for a fixed-size window (moving average over the last N periods); `.expanding()` when the window should grow to include everything from the start (a running total/cumulative stat).

## Summary

- `.agg()` collapses to one row per group; `.transform()` broadcasts a per-group value back to every row; `.filter()` keeps/drops whole groups.
- `groupby(...).rank(method=...)` maps directly onto SQL/Spark's `ROW_NUMBER`/`RANK`/`DENSE_RANK`.
- Top-N-per-group = rank within group + filter, same shape as the Spark solution in `../../spark_practice/`.
- Next: `05_performance_and_scaling.ipynb`.